# Evaluation Metrics for RAG Systems

## Overview

This notebook covers evaluation metrics for measuring RAG system performance, including relevance, faithfulness, and answer quality.

## Learning Objectives

- Understand key RAG evaluation metrics
- Implement relevance and faithfulness scoring
- Calculate answer quality metrics (BLEU, ROUGE)
- Measure retrieval performance
- Apply NVIDIA Agent Intelligence Toolkit for evaluation

## Exam Objectives: 3.1, 3.2, 3.3
## Estimated Time: 90-105 minutes

## 🎯 Exam Focus

This notebook covers concepts that appear frequently on the NCP-AAI exam:

**High-Priority Topics:**
- ⭐⭐⭐ Evaluation metrics (precision, recall, F1, faithfulness)
- ⭐⭐⭐ A/B testing frameworks
- ⭐⭐ Parameter tuning trade-offs
- ⭐⭐ Cost-performance optimization

**Common Exam Scenarios:**
- Selecting appropriate metrics for different agent types
- Designing A/B tests for agent improvements
- Balancing accuracy vs latency vs cost

**Key Concepts to Master:**
- RAG-specific metrics (faithfulness, relevance)
- Statistical significance in A/B testing
- Accuracy-latency trade-offs

## Setup: Import Dependencies

In [ ]:
# Import core libraries for evaluation
import os
import sys
from typing import Dict, List, Any, Tuple
import numpy as np
from collections import Counter

# LangChain imports
try:
    import langchain
    print(f"✅ LangChain version: {langchain.__version__}")
except ImportError:
    print("⚠️  LangChain not available")

print("🎯 Setup complete!")

## Theory: RAG Evaluation Metrics

### Why Evaluate RAG Systems?

Evaluation is critical for:
- **Quality assurance**: Ensure system meets requirements
- **Optimization**: Identify areas for improvement
- **Comparison**: Choose between different approaches
- **Monitoring**: Track performance over time

### Key Metric Categories

#### 1. Retrieval Metrics
Measure how well the system retrieves relevant documents:
- **Precision**: Fraction of retrieved docs that are relevant
- **Recall**: Fraction of relevant docs that are retrieved
- **F1 Score**: Harmonic mean of precision and recall
- **MRR (Mean Reciprocal Rank)**: Average of reciprocal ranks
- **NDCG (Normalized Discounted Cumulative Gain)**: Ranking quality

#### 2. Generation Metrics
Measure quality of generated answers:
- **BLEU**: N-gram overlap with reference
- **ROUGE**: Recall-oriented overlap
- **METEOR**: Semantic similarity
- **BERTScore**: Contextual embedding similarity

#### 3. RAG-Specific Metrics
Measure RAG system properties:
- **Faithfulness**: Answer grounded in retrieved docs
- **Answer Relevance**: Answer addresses the question
- **Context Relevance**: Retrieved docs relevant to question

### Metric Formulas

**Precision**: P = TP / (TP + FP)
**Recall**: R = TP / (TP + FN)
**F1**: F1 = 2 * (P * R) / (P + R)

Where:
- TP = True Positives (relevant docs retrieved)
- FP = False Positives (irrelevant docs retrieved)
- FN = False Negatives (relevant docs not retrieved)

## Implementation: Retrieval Metrics

Let's implement core retrieval evaluation metrics.

In [ ]:
# Retrieval metrics implementation
class RetrievalMetrics:
    """
    Calculate retrieval performance metrics.
    
    Metrics:
    - Precision: Accuracy of retrieved documents
    - Recall: Coverage of relevant documents
    - F1 Score: Balance between precision and recall
    - Mean Reciprocal Rank (MRR): Ranking quality
    """
    
    @staticmethod
    def precision(retrieved: List[str], relevant: List[str]) -> float:
        """
        Calculate precision: fraction of retrieved docs that are relevant.
        
        Args:
            retrieved: List of retrieved document IDs
            relevant: List of relevant document IDs
        
        Returns:
            Precision score (0.0 to 1.0)
        """
        if not retrieved:
            return 0.0
        
        # Count how many retrieved docs are relevant
        relevant_set = set(relevant)
        true_positives = sum(1 for doc in retrieved if doc in relevant_set)
        
        return true_positives / len(retrieved)
    
    @staticmethod
    def recall(retrieved: List[str], relevant: List[str]) -> float:
        """
        Calculate recall: fraction of relevant docs that are retrieved.
        
        Args:
            retrieved: List of retrieved document IDs
            relevant: List of relevant document IDs
        
        Returns:
            Recall score (0.0 to 1.0)
        """
        if not relevant:
            return 0.0
        
        # Count how many relevant docs were retrieved
        retrieved_set = set(retrieved)
        true_positives = sum(1 for doc in relevant if doc in retrieved_set)
        
        return true_positives / len(relevant)
    
    @staticmethod
    def f1_score(retrieved: List[str], relevant: List[str]) -> float:
        """
        Calculate F1 score: harmonic mean of precision and recall.
        
        Args:
            retrieved: List of retrieved document IDs
            relevant: List of relevant document IDs
        
        Returns:
            F1 score (0.0 to 1.0)
        """
        precision = RetrievalMetrics.precision(retrieved, relevant)
        recall = RetrievalMetrics.recall(retrieved, relevant)
        
        # Handle edge case where both are zero
        if precision + recall == 0:
            return 0.0
        
        return 2 * (precision * recall) / (precision + recall)
    
    @staticmethod
    def mean_reciprocal_rank(retrieved_lists: List[List[str]], relevant_lists: List[List[str]]) -> float:
        """
        Calculate Mean Reciprocal Rank (MRR).
        
        MRR measures how high the first relevant document appears in rankings.
        
        Args:
            retrieved_lists: List of retrieved document lists (one per query)
            relevant_lists: List of relevant document lists (one per query)
        
        Returns:
            MRR score (0.0 to 1.0)
        """
        reciprocal_ranks = []
        
        for retrieved, relevant in zip(retrieved_lists, relevant_lists):
            relevant_set = set(relevant)
            
            # Find rank of first relevant document
            for rank, doc in enumerate(retrieved, start=1):
                if doc in relevant_set:
                    reciprocal_ranks.append(1.0 / rank)
                    break
            else:
                # No relevant document found
                reciprocal_ranks.append(0.0)
        
        return np.mean(reciprocal_ranks) if reciprocal_ranks else 0.0

# Test retrieval metrics
print("=== Testing Retrieval Metrics ===")

# Example: Retrieved vs relevant documents
retrieved_docs = ["doc1", "doc2", "doc3", "doc4", "doc5"]
relevant_docs = ["doc1", "doc3", "doc6", "doc7"]

print(f"\nRetrieved: {retrieved_docs}")
print(f"Relevant: {relevant_docs}")
print(f"\nMetrics:")
print(f"  Precision: {RetrievalMetrics.precision(retrieved_docs, relevant_docs):.3f}")
print(f"  Recall: {RetrievalMetrics.recall(retrieved_docs, relevant_docs):.3f}")
print(f"  F1 Score: {RetrievalMetrics.f1_score(retrieved_docs, relevant_docs):.3f}")

# Test MRR with multiple queries
print("\n=== Testing Mean Reciprocal Rank ===")
retrieved_lists = [
    ["doc1", "doc2", "doc3"],  # Query 1: relevant doc at position 1
    ["doc4", "doc5", "doc6"],  # Query 2: relevant doc at position 3
    ["doc7", "doc8", "doc9"]   # Query 3: no relevant doc
]
relevant_lists = [
    ["doc1"],
    ["doc6"],
    ["doc10"]
]

mrr = RetrievalMetrics.mean_reciprocal_rank(retrieved_lists, relevant_lists)
print(f"MRR: {mrr:.3f}")
print(f"Calculation: (1/1 + 1/3 + 0) / 3 = {(1.0 + 1.0/3 + 0) / 3:.3f}")

## References

### Course Materials

**Course Notes:** [Module 3: Evaluation and Tuning](../../course-notes/module-03-evaluation-tuning.md)

### Practice Questions

**Exam Questions:** [Practice Questions](../../exam-questions/domain-03-evaluation.md)

### Hands-On Labs

- [Lab 4: Evaluation and Optimization](../../labs/lab-04-evaluation-optimization/README.md)

### Additional Resources

- [NVIDIA NeMo Documentation](https://docs.nvidia.com/nemo/)
- [LangChain Documentation](https://python.langchain.com/)
- [NVIDIA AI Endpoints](https://build.nvidia.com/)
